# Feature e classificazione — esperimento completo

Eseguire in Jupyter con le dipendenze del corso e UrbanSound8K disponibile. Usare Run → Run All Cells. PyTorch e il dataset completo sono richiesti per le demo della CNN.

In [ ]:
from pathlib import Path
from collections import Counter
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
from IPython.display import Audio, display
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "lab-2026/audiolab").is_dir())
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
FS = al.URBAN_CONFIG["sr"]
OUTPUT = ROOT / "output/classificazione"
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.2})

In [ ]:
cartella = Path(os.environ.get("URBANSOUND8K", str(ROOT / "datasets/UrbanSound8K")))
records = al.urban_metadata(cartella)
split = al.split_dataset(records)
for nome, righe in split.items():
    print(nome, len(righe), "clip; per classe:", dict(sorted(Counter(r["label"] for r in righe).items())))
print("Esempio:", Path(split["train"][0]["file"]).name)
al.listen(al.load_clip(split["train"][0]["file"], sr=FS), FS)

In [ ]:
rng = np.random.default_rng(0)
t = np.arange(FS)/FS
segnali = {
    "Tono 330 Hz": al.sine(330, 1, FS, amplitude=1.0),
    "Somma armonica": al.sum_harmonics(range(1,10), f0=330, duration=1, sr=FS),
    "Rumore": al.white_noise(len(t),seed=0),
}
fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for ax, (nome, y) in zip(axes, segnali.items()):
    y = y/al.rms(y)*0.05
    segnali[nome] = y
    A = np.abs(al.analysis.stft(y, n_fft=1024, hop_length=512))
    P = A**2
    centroide = al.analysis.feature.spectral_centroid(S=A, sr=FS).mean()
    rolloff = al.analysis.feature.spectral_rolloff(S=P, sr=FS, roll_percent=0.85).mean()
    flat = al.analysis.feature.spectral_flatness(S=P, power=1.0).mean()
    zcr = al.analysis.feature.zero_crossing_rate(y).mean()
    print(f"{nome}: RMS 0.050; ZCR {zcr:.3f}; centroide {centroide:.0f} Hz; "
          f"rolloff {rolloff:.0f} Hz; flatness {flat:.3f}")
    ax.plot(al.analysis.fft_frequencies(sr=FS, n_fft=1024), A.mean(axis=1))
    ax.set_ylabel(nome); ax.set_xlim(0, FS/2)
axes[-1].set_xlabel("Frequenza [Hz]")
fig.suptitle("Spettro medio in ampiezza: stessa scala di frequenza")
plt.tight_layout(); plt.show()

In [ ]:
y = al.load_clip(split["train"][0]["file"], sr=FS)
_, potenza = al.stft_power(y, FS, n_fft=1024, hop=512)
A = np.sqrt(potenza)
B = al.mel_filterbank(FS, n_fft=1024, n_mels=64, htk=False, norm="slaney")
mel = B @ A**2
logmel = al.log_mel(y, FS, n_mels=64, n_fft=1024, hop=512)
mfcc = al.mfcc_matrix(y, FS, n_mfcc=13, n_mels=64, n_fft=1024, hop=512)
print("Banco:", B.shape, "potenza:", A.shape, "log-mel:", logmel.shape, "MFCC:", mfcc.shape)
fig, ax = plt.subplots(figsize=(9, 3))
f = al.analysis.fft_frequencies(sr=FS, n_fft=1024)
for banda in B[::4]: ax.plot(f, banda)
ax.set(xlabel="Frequenza [Hz]", ylabel="Peso", title="Una banda ogni quattro: banco Slaney normalizzato per area")
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(3, 1, figsize=(10, 8))
for ax, S, asse_y, titolo in zip(axes,
    [al.analysis.power_to_db(A**2, ref=1.0), logmel, mfcc],
    ["linear", "mel", None], ["Potenza STFT in dB", "Log-mel: 64 bande", "MFCC: coefficienti 0–12"]):
    img = al.analysis.display.specshow(S, sr=FS, hop_length=512, x_axis="time",
                                  y_axis=asse_y, ax=ax)
    ax.set_title(titolo)
    fig.colorbar(img, ax=ax)
axes[-1].set_ylabel("Indice del coefficiente")
plt.tight_layout(); plt.show()

In [ ]:
permutati = mfcc[:, np.random.default_rng(7).permutation(mfcc.shape[1])]
print("Media invariata:", np.allclose(mfcc.mean(1), permutati.mean(1)))
print("Deviazione standard invariata:", np.allclose(mfcc.std(1), permutati.std(1)))

In [ ]:
dati, chiave = al.prepare_features(split, OUTPUT)
modelli = {}
costi = {}
for nome, dimensioni in (("Dummy", 13), ("RF media", 13), ("RF media+std", 26)):
    modello = (DummyClassifier(strategy="most_frequent") if nome == "Dummy" else
               RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=2))
    inizio = time.perf_counter()
    modello.fit(dati["train_feat"][:, :dimensioni], dati["train_y"])
    costi[nome] = time.perf_counter()-inizio
    pred = modello.predict(dati["val_feat"][:, :dimensioni])
    print(nome, "macro-F1 validazione:",
          round(f1_score(dati["val_y"], pred, labels=range(10), average="macro", zero_division=0), 3))
    modelli[nome] = (modello, dimensioni)

In [ ]:
torch.manual_seed(42)
rete = al.build_sound_cnn()
x = torch.zeros(1, 1, 64, 173)
with torch.no_grad():
    for i, blocco in enumerate(rete.blocks, 1):
        x = blocco(x); print("Blocco", i, tuple(x.shape))
print("Parametri:", sum(p.numel() for p in rete.parameters()))

In [ ]:
media = float(dati["train_mel"].mean())
std = max(float(dati["train_mel"].std()), 1e-6)
X = {nome: ((dati[f"{nome}_mel"]-media)/std)[:, None].astype(np.float32)
     for nome in ("train", "val", "test")}
rete, info = al.train_or_load_cnn(X["train"], dati["train_y"], X["val"], dati["val_y"],
                               OUTPUT, chiave, epochs=8, seed=42)
print("Epoca scelta sulla validazione:", info["best_epoch"])
print("Tempo dell'addestramento registrato:", round(info["seconds"], 1), "s")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
storia = info["history"]
axes[0].plot([r["epoch"] for r in storia], [r["loss"] for r in storia])
axes[0].set(title="Loss train", xlabel="Epoca")
for campo in ("train_f1", "val_f1"):
    axes[1].plot([r["epoch"] for r in storia], [r[campo] for r in storia], label=campo)
axes[1].set(title="Macro-F1", xlabel="Epoca", ylim=(0, 1)); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
predizioni = {}
for nome, (modello, dimensioni) in modelli.items():
    predizioni[nome] = modello.predict(dati["test_feat"][:, :dimensioni])
predizioni["CNN"] = al.predict(rete, X["test"])
veri = dati["test_y"]
for nome, pred in predizioni.items():
    print(f"{nome:14s} accuratezza {accuracy_score(veri, pred):.3f}; "
          f"macro-F1 {f1_score(veri, pred, labels=range(10), average='macro', zero_division=0):.3f}")
print("Tempi fit RF/Dummy (s):", {k:round(v, 2) for k,v in costi.items()})
print("Tempo training CNN (s):", round(info["seconds"], 2))
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, nome in zip(axes, ("RF media+std", "CNN")):
    ConfusionMatrixDisplay.from_predictions(veri, predizioni[nome], labels=range(10),
        display_labels=al.URBAN_CLASSES, normalize="true", values_format=".1f", cmap="Blues",
        xticks_rotation=90, colorbar=False, im_kw={"vmin": 0, "vmax": 1}, ax=ax)
    ax.set_title(nome + " — righe normalizzate")
    ax.set_xlabel("Classe predetta"); ax.set_ylabel("Classe reale")
plt.tight_layout(); plt.show()

errori = np.flatnonzero(predizioni["CNN"] != veri)
for k in errori[:3]:
    record = split["test"][k]
    print(Path(record["file"]).name, "vero:", al.URBAN_CLASSES[veri[k]],
          "predetto:", al.URBAN_CLASSES[predizioni["CNN"][k]])
    al.listen(al.load_clip(record["file"], sr=FS), FS)  # il file originale, non lo spettrogramma!

In [ ]:
y = al.load_clip(split["val"][0]["file"], sr=FS)
rng = np.random.default_rng(11)
rms = np.sqrt(np.mean(y*y))
rumore = rng.standard_normal(len(y))
rumore = rumore/np.sqrt(np.mean(rumore**2))*rms*10**(-10/20)  # SNR 10 dB
_, f_originale = al.representations(y)
_, f_rumoroso = al.representations(y+rumore)
modello, _ = modelli["RF media+std"]
for nome, f in (("Originale", f_originale), ("Con rumore", f_rumoroso)):
    print(nome, "→", al.URBAN_CLASSES[modello.predict(f[None])[0]])
print("Distanza fra vettori MFCC:", np.linalg.norm(f_originale-f_rumoroso))

In [ ]:
t = np.arange(FS)/FS
accordo = sum(np.sin(2*np.pi*librosa.midi_to_hz(m)*t) for m in (60, 64, 67))/6
profilo = librosa.feature.chroma_stft(y=accordo, sr=FS).mean(axis=1)
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"], profilo)
ax.set(title="Accordo sintetico Do–Mi–Sol", ylabel="Chroma medio")
plt.tight_layout(); plt.show()